In [0]:
# On Databricks, spark already exists — we configure it
# Create 200 partitions for shuffling 
spark.conf.set("spark.sql.shuffle.partitions", "200")

# AQE (Adaptive Query Execution) is not supported in Spark Connect, so we skip enabling it.
# spark.conf.set("spark.sql.adaptive.enabled", "true")

# KryoSerializer is not supported in Spark Connect, so we skip setting spark.serializer.
# spark.conf.set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")

# Snappy compression is not supported in Spark Connect, so we skip setting spark.sql.parquet.compression.codec.
# spark.conf.set("spark.sql.parquet.compression.codec", "snappy")

# Arrow optimization is not supported in Spark Connect, so we skip setting spark.sql.execution.arrow.pyspark.enabled.
# spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

# GPU Scalability hooks (Enterprise Edition only)
    # spark.conf.set("spark.rapids.sql.enabled", "true")
    # spark.conf.set("spark.plugins", "com.nvidia.spark.SQLPlugin")
    # spark.conf.set("spark.rapids.sql.concurrentGpuTasks", "2")


print("✅ Spark version:", spark.version)
# Commented out as Spark UI is not supported in the community edition
#print("✅ Spark UI:", spark.sparkContext.uiWebUrl)

✅ Spark version: 4.1.0


In [0]:
# ═══════════════════════════════════════════════════════════
# CELL 2: MLFLOW EXPERIMENT TRACKING
# ═══════════════════════════════════════════════════════════

#MLflow tracks every experiment: which model, what settings, what score. Like a lab notebook that writes itself.
import mlflow
import mlflow.spark

mlflow.set_experiment("/Users/aworetanoluwole77@gmail.com/FEMA_FloodGuard")

print("✅ MLflow tracking URI:", mlflow.get_tracking_uri())

✅ MLflow tracking URI: databricks


In [0]:
# ═══════════════════════════════════════════════════════════
# CELL 4: BRONZE — DATAFRAME-BASED INGESTION (Spark Connect compatible)
#   df = spark.read.csv(...)
#   df = df.filter(...)
#   df = df.select(...)
# Note: DBFS root is disabled in this workspace. Use /Workspace/ or S3 paths instead.
# IMPORTANT: Do not use the parent directory containing multiple datasets/partitioned outputs.
#            Point directly to the CSV file to avoid directory structure conflicts.
# FIX: Use the correct Unity Catalog volume path, no dbfs:/ prefix.
# ═══════════════════════════════════════════════════════════

# Read CSV as DataFrame, inferring schema and header
raw_df = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("mode", "DROPMALFORMED")
        .csv("/Volumes/workspace/default/fema_claims_project/FimaNfipClaims.csv")
)

# Drop rows with too few columns (less than 10 non-null values)
from pyspark.sql.functions import col
raw_df = raw_df.dropna(thresh=10)

print(f"✅ Bronze DataFrame: {raw_df.count():,} rows × {len(raw_df.columns)} columns")
raw_df.printSchema()

✅ Bronze DataFrame: 2,719,735 rows × 73 columns
root
 |-- agricultureStructureIndicator: integer (nullable = true)
 |-- asOfDate: timestamp (nullable = true)
 |-- basementEnclosureCrawlspaceType: integer (nullable = true)
 |-- policyCount: integer (nullable = true)
 |-- crsClassificationCode: integer (nullable = true)
 |-- dateOfLoss: timestamp (nullable = true)
 |-- elevatedBuildingIndicator: integer (nullable = true)
 |-- elevationCertificateIndicator: string (nullable = true)
 |-- elevationDifference: double (nullable = true)
 |-- baseFloodElevation: double (nullable = true)
 |-- ratedFloodZone: string (nullable = true)
 |-- houseWorship: integer (nullable = true)
 |-- locationOfContents: integer (nullable = true)
 |-- lowestAdjacentGrade: double (nullable = true)
 |-- lowestFloorElevation: double (nullable = true)
 |-- numberOfFloorsInTheInsuredBuilding: integer (nullable = true)
 |-- nonProfitIndicator: integer (nullable = true)
 |-- obstructionType: integer (nullable = true)
 |--

In [0]:
# Save Bronze as Parquet to Unity Catalog volume (DBFS root is disabled)
raw_df.write.mode("overwrite").parquet("/Volumes/workspace/default/fema_claims_project/fema_raw.parquet")
print("✅ Bronze Parquet saved!")

✅ Bronze Parquet saved!
